In [1]:
import os
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

from catboost import CatBoostClassifier, Pool
import torch
from itertools import product

In [2]:
# =========================
# Config
# =========================
class CFG:
    SEED = 42
    N_SPLITS = 5
    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"

    SAVE_OOF = True
    SAVE_PRED = True
    SAVE_SUB = True

    MODEL_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": 3000,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 5.0,
        "random_strength": 1.0,
        "bagging_temperature": 0.0,
        "grow_policy": "SymmetricTree",
        "bootstrap_type": "Bayesian",
        "random_seed": SEED,
        "verbose": 200,
        "early_stopping_rounds": 200,
        "task_type": "GPU" if torch.cuda.is_available() else "CPU",
    }

In [3]:
# =========================
# Column groups
# =========================
NUM_COLS_BASE = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

CAT_COLS_BASE = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
]

BIN_COLS_BASE = [
    "Mulching_Used",
]

CAT_ALL_BASE = CAT_COLS_BASE + BIN_COLS_BASE
FEATURE_COLS_BASE = NUM_COLS_BASE + CAT_COLS_BASE + BIN_COLS_BASE

In [4]:
# =========================
# Utilities
# =========================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def load_data():
    train_df = pd.read_csv(CFG.TRAIN_PATH)
    test_df = pd.read_csv(CFG.TEST_PATH)
    return train_df, test_df


def validate_columns(train_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
    expected_train = set(FEATURE_COLS_BASE + [CFG.TARGET_COL, CFG.ID_COL])
    expected_test = set(FEATURE_COLS_BASE + [CFG.ID_COL])

    train_missing = expected_train - set(train_df.columns)
    train_extra = set(train_df.columns) - expected_train

    test_missing = expected_test - set(test_df.columns)
    test_extra = set(test_df.columns) - expected_test

    print("train_missing:", train_missing)
    print("train_extra:", train_extra)
    print("test_missing:", test_missing)
    print("test_extra:", test_extra)

    if train_missing or test_missing:
        raise ValueError("Column mismatch detected.")


def cast_dtypes(train_df: pd.DataFrame, test_df: pd.DataFrame):
    # CatBoostに素直に渡すため、カテゴリ系は string に寄せる
    for col in CAT_ALL_BASE:
        train_df[col] = train_df[col].astype(str)
        test_df[col] = test_df[col].astype(str)

    # 数値は数値のまま
    for col in NUM_COLS_BASE:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

    return train_df, test_df


def make_pools(X_tr, y_tr, X_va, y_va, X_te):
    train_pool = Pool(
        data=X_tr,
        label=y_tr,
        cat_features=CAT_ALL_BASE,
    )
    valid_pool = Pool(
        data=X_va,
        label=y_va,
        cat_features=CAT_ALL_BASE,
    )
    test_pool = Pool(
        data=X_te,
        cat_features=CAT_ALL_BASE,
    )
    return train_pool, valid_pool, test_pool


def predict_with_bias(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    adjusted = proba + bias.reshape(1, -1)
    return np.argmax(adjusted, axis=1)


def evaluate_per_class_recall(y_true: np.ndarray, y_pred: np.ndarray, class_names=None):
    n_classes = len(np.unique(y_true))
    recalls = {}
    for cls_idx in range(n_classes):
        mask = (y_true == cls_idx)
        recall = float((y_pred[mask] == y_true[mask]).mean()) if mask.sum() > 0 else np.nan
        key = class_names[cls_idx] if class_names is not None else cls_idx
        recalls[key] = recall
    return recalls


def search_topk_class_bias(
    oof_proba: np.ndarray,
    y_true: np.ndarray,
    bias_grid=None,
    fix_first_class_zero=True,
    top_k=10,
    verbose=True,
):
    """
    粗探索して上位top_k個の bias 候補を返す
    """
    n_classes = oof_proba.shape[1]

    if bias_grid is None:
        bias_grid = [-0.60, -0.50, -0.40, -0.30, -0.20, -0.10, -0.05,
                     0.0,
                     0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60]

    results = []

    if fix_first_class_zero:
        for rest in product(bias_grid, repeat=n_classes - 1):
            bias = np.array([0.0] + list(rest), dtype=np.float32)
            pred = predict_with_bias(oof_proba, bias)
            score = balanced_accuracy_score(y_true, pred)
            results.append((score, bias.copy()))
    else:
        for bias_tuple in product(bias_grid, repeat=n_classes):
            bias = np.array(bias_tuple, dtype=np.float32)
            pred = predict_with_bias(oof_proba, bias)
            score = balanced_accuracy_score(y_true, pred)
            results.append((score, bias.copy()))

    results.sort(key=lambda x: x[0], reverse=True)
    top_results = results[:top_k]

    if verbose:
        print("Top coarse candidates:")
        for i, (score, bias) in enumerate(top_results, 1):
            print(f"{i:2d}: score={score:.6f}, bias={bias}")

    return top_results


def refine_from_topk_candidates(
    oof_proba: np.ndarray,
    y_true: np.ndarray,
    top_results,
    step=0.02,
    radius=0.10,
    fix_first_class_zero=True,
    verbose=True,
):
    """
    coarse top候補の近傍を細探索して最良を返す
    """
    n_classes = oof_proba.shape[1]
    best_score = -1.0
    best_bias = None

    for coarse_score, coarse_bias in top_results:
        grids = []
        for i in range(n_classes):
            if fix_first_class_zero and i == 0:
                grids.append([0.0])
            else:
                center = coarse_bias[i]
                vals = np.arange(center - radius, center + radius + 1e-12, step)
                vals = np.round(vals, 6)
                grids.append(vals.tolist())

        for bias_tuple in product(*grids):
            bias = np.array(bias_tuple, dtype=np.float32)
            pred = predict_with_bias(oof_proba, bias)
            score = balanced_accuracy_score(y_true, pred)

            if score > best_score:
                best_score = score
                best_bias = bias.copy()

    if verbose:
        print("Refined best:")
        print(f"best_score={best_score:.6f}")
        print(f"best_bias ={best_bias}")

    return best_bias, best_score

In [5]:
# =========================
# Main CV
# =========================

seed_everything(CFG.SEED)

train_df, test_df = load_data()
validate_columns(train_df, test_df)
train_df, test_df = cast_dtypes(train_df, test_df)

X = train_df[FEATURE_COLS_BASE].copy()
X_test = test_df[FEATURE_COLS_BASE].copy()
y_raw = train_df[CFG.TARGET_COL].copy()

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = list(le.classes_)
n_classes = len(class_names)

print("classes:", class_names)
print("train shape:", train_df.shape)
print("test shape:", test_df.shape)

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

oof_proba = np.zeros((len(train_df), n_classes), dtype=np.float32)
test_proba = np.zeros((len(test_df), n_classes), dtype=np.float32)
fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("=" * 60)
    print(f"fold {fold} / {CFG.N_SPLITS}")
    print("=" * 60)

    X_tr = X.iloc[tr_idx].copy()
    X_va = X.iloc[va_idx].copy()
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    train_pool, valid_pool, test_pool = make_pools(X_tr, y_tr, X_va, y_va, X_test)

    model = CatBoostClassifier(**CFG.MODEL_PARAMS)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
    )

    va_proba = model.predict_proba(valid_pool)
    te_proba = model.predict_proba(test_pool)

    oof_proba[va_idx] = va_proba
    test_proba += te_proba / CFG.N_SPLITS

    va_pred = np.argmax(va_proba, axis=1)
    fold_bacc = balanced_accuracy_score(y_va, va_pred)
    fold_scores.append(fold_bacc)

    print(f"fold {fold} balanced_accuracy = {fold_bacc:.6f}")

print("=" * 60)
print("fold scores:", [round(s, 6) for s in fold_scores])
print(f"cv mean balanced_accuracy = {np.mean(fold_scores):.6f}")
print(f"cv std  balanced_accuracy = {np.std(fold_scores):.6f}")

# =========================
# class bias optimization
# =========================
print("=" * 60)
print("Bias optimization (expanded search)")

# biasなし baseline
oof_pred_base = np.argmax(oof_proba, axis=1)
oof_bacc_base = balanced_accuracy_score(y, oof_pred_base)
print(f"OOF balanced_accuracy (no bias) = {oof_bacc_base:.6f}")

recalls_base = evaluate_per_class_recall(y, oof_pred_base, class_names)
print("Per-class recall (no bias):")
for k, v in recalls_base.items():
    print(f"  {k}: {v:.6f}")

# 1) 粗探索レンジを広げる
coarse_grid = [
    -0.60, -0.50, -0.40, -0.30, -0.20, -0.10, -0.05,
     0.00,
     0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60
]

top_results = search_topk_class_bias(
    oof_proba=oof_proba,
    y_true=y,
    bias_grid=coarse_grid,
    fix_first_class_zero=True,
    top_k=12,
    verbose=True,
)

# 2) 上位候補近傍を細探索
best_bias, best_bias_score = refine_from_topk_candidates(
    oof_proba=oof_proba,
    y_true=y,
    top_results=top_results,
    step=0.01,
    radius=0.08,
    fix_first_class_zero=True,
    verbose=True,
)

# 3) bias適用後 OOF
oof_pred_bias = predict_with_bias(oof_proba, best_bias)
oof_bacc_bias = balanced_accuracy_score(y, oof_pred_bias)
print(f"\nOOF balanced_accuracy (with bias) = {oof_bacc_bias:.6f}")

recalls_bias = evaluate_per_class_recall(y, oof_pred_bias, class_names)
print("Per-class recall (with bias):")
for k, v in recalls_bias.items():
    print(f"  {k}: {v:.6f}")

print("\nDelta:")
print(f"OOF gain = {oof_bacc_bias - oof_bacc_base:+.6f}")
for cls in class_names:
    print(f"{cls} recall gain = {recalls_bias[cls] - recalls_base[cls]:+.6f}")

# 保存
# best bias 保存
np.save("best_class_bias_wide.npy", best_bias)

# test 側にも同じ bias を適用
test_pred_bias = predict_with_bias(test_proba, best_bias)
test_label_bias = le.inverse_transform(test_pred_bias)

sub_bias_df = pd.DataFrame({
    CFG.ID_COL: test_df[CFG.ID_COL],
    CFG.TARGET_COL: test_label_bias
})

sub_bias_df.to_csv("submission_catboost_strict_baseline_bias_wide.csv", index=False)

# 必要なら proba も bias 実験名で保存
np.save("oof_catboost_strict_baseline_bias_wide_proba.npy", oof_proba)
np.save("pred_catboost_strict_baseline_bias_wide_proba.npy", test_proba)

print("\nSaved:")
print("- best_class_bias_wide.npy")
print("- submission_catboost_strict_baseline_bias_wide.csv")
print("- oof_catboost_strict_baseline_bias_wide_proba.npy")
print("- pred_catboost_strict_baseline_bias_wide_proba.npy")

train_missing: set()
train_extra: set()
test_missing: set()
test_extra: set()
classes: ['High', 'Low', 'Medium']
train shape: (630000, 21)
test shape: (270000, 20)
fold 1 / 5
0:	learn: 1.0374401	test: 1.0373987	best: 1.0373987 (0)	total: 116ms	remaining: 5m 47s
200:	learn: 0.0642986	test: 0.0645448	best: 0.0645448 (200)	total: 2.98s	remaining: 41.6s
400:	learn: 0.0628174	test: 0.0634690	best: 0.0634690 (400)	total: 5.57s	remaining: 36.1s
600:	learn: 0.0613091	test: 0.0625561	best: 0.0625561 (600)	total: 7.97s	remaining: 31.8s
800:	learn: 0.0598289	test: 0.0618103	best: 0.0618103 (800)	total: 10.4s	remaining: 28.5s
1000:	learn: 0.0587111	test: 0.0613316	best: 0.0613316 (1000)	total: 13s	remaining: 26s
1200:	learn: 0.0578623	test: 0.0609944	best: 0.0609944 (1200)	total: 15.7s	remaining: 23.5s
1400:	learn: 0.0569908	test: 0.0607146	best: 0.0607146 (1400)	total: 18.3s	remaining: 20.9s
1600:	learn: 0.0561740	test: 0.0604560	best: 0.0604542 (1597)	total: 20.9s	remaining: 18.3s
1800:	learn: 0

In [6]:
loaded_bias = np.load("/kaggle/working/best_class_bias_wide.npy")
print("le.classes_:", le.classes_)
print("best_bias  :", loaded_bias)

for cls, bias in zip(le.classes_, loaded_bias):
    print(f"{cls}: {bias:.6f}")

le.classes_: ['High' 'Low' 'Medium']
best_bias  : [ 0.   -0.32 -0.68]
High: 0.000000
Low: -0.320000
Medium: -0.680000
